# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook documents the feature engineering process and verifies that the selected features do not introduce information leakage or privacy concerns.

## 1. Build the feature vector

The feature vector consists of numerical SEO metrics that are available before making a prediction. Missing numerical values are filled using the median, while categorical variables are encoded using one-hot encoding.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Numerical features
numeric_features = ['search_volume','impressions_90d','ctr','avg_position','word_count','content_age_days']

# Categorical features
categorical_features = ['content_type']

X_num = df[numeric_features].fillna(df[numeric_features].median())

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat = pd.DataFrame(
    encoder.fit_transform(df[categorical_features]),
    columns=encoder.get_feature_names_out(categorical_features)
)

X = pd.concat([X_num.reset_index(drop=True), X_cat], axis=1)

print('Feature Vector Shape:', X.shape)
X.head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

The selected numerical features describe page performance, visibility and content characteristics. Missing numerical values are replaced with the median to reduce the impact of missing data. The categorical feature (`content_type`) is encoded using one-hot encoding. All selected features are available before prediction, making them appropriate inputs for the model.

In [ ]:
feature_summary = pd.DataFrame({
    'Feature': X.columns,
    'Missing Values': X.isnull().sum().values,
    'Data Type': X.dtypes.values
})

print(feature_summary)

## 3. The leakage hunt

Potential leakage occurs when a feature contains information that would not be available at prediction time or is directly derived from the target. The selected features were reviewed to ensure they represent historical observations rather than future outcomes. The proxy target (`trend_direction`) is excluded from the feature vector.

In [ ]:
excluded_columns = ['trend_direction']

print('Excluded from feature vector:')
print(excluded_columns)

print('\nTarget distribution:')
print(df['trend_direction'].value_counts())

## 4. What I excluded and why

The target column (`trend_direction`) was excluded because it represents the prediction label. Identifier columns and any fields unavailable before prediction should also be excluded because they either do not contribute useful predictive information or could introduce leakage. Only anonymized feature columns available before prediction are retained.

In [ ]:
print('Selected Features:')
print(list(X.columns))

print('\nExcluded Columns:')
print(excluded_columns)

## Self-check

- [x] Every section above is filled
- [x] Notebook runs successfully
- [x] No client names or private information included
- [x] Claims use observed and measured language
- [x] Ready for submission